# Orchestrazione della pipeline Media Cloud

Questo notebook e' la guida operativa della pipeline: non contiene logica di scraping o di classificazione duplicata. Ogni fase richiama uno script Python versionato, ne mostra il log e verifica gli output attesi.

**Eseguire le celle nell'ordine.** Per sicurezza tutte le fasi sono disattivate all'avvio: attivarne una alla volta nella sezione Configurazione.

## Flusso e responsabilita'

1. **Discovery** — `mediacloud_spike.py` interroga Media Cloud e salva gli URL con metadati. Richiede `MC_API_KEY`.
2. **Full-text** — `mediacloud_fulltext.py` scarica le pagine, estrae il testo, rileva la lingua e filtra il rumore sportivo della Lega.
3. **Classificazione** — `news_topic_model.py` esegue TF-IDF + NMF e produce CSV per la validazione umana dei macrotemi.

Gli script sono la fonte di verita' perche' possono essere testati, eseguiti da terminale e in futuro orchestrati da Dagster. Il notebook e' invece la superficie adatta per run manuali, demo e controllo dei risultati.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    raise RuntimeError('Aprire il notebook dal repository temi-politici.')

PYTHON = sys.executable
URLS_FILE = ROOT / 'data' / 'raw' / 'mediacloud_urls.jsonl'
FULLTEXT_FILE = ROOT / 'data' / 'raw' / 'mediacloud_fulltext.jsonl'
COVERAGE_FILE = ROOT / 'data' / 'processed' / 'mediacloud_coverage.csv'
REVIEW_FILE = ROOT / 'data' / 'processed' / 'news_topic_review.csv'
TERMS_FILE = ROOT / 'data' / 'processed' / 'news_topic_terms.csv'


## Configurazione del run

Impostare `True` **solo** sulla fase che si vuole eseguire, quindi lanciare la relativa cella piu' sotto. Il primo test usa 500 URL: dopo aver controllato qualità, tempi e spazio, impostare `DISCOVERY_MAX_STORIES = None` per l'intero periodo di sei mesi.

In [ ]:
RUN_DISCOVERY = False
RUN_FULLTEXT = False
RUN_CLASSIFIER = False

DISCOVERY_MAX_STORIES = 500  # None = tutto il periodo; usare dopo lo smoke test
N_TOPICS = 12


## Preflight

Questa cella non avvia nulla: controlla solo prerequisiti e stato locale. La chiave non viene mai stampata. Se si usa un file `.env` locale, deve contenere `MC_API_KEY=...`; il file e' ignorato da Git.

In [ ]:
def local_drive_root():
    configured = os.environ.get('GOOGLE_DRIVE_EXPORT_DIR')
    local_file = ROOT / '.drive-export-dir'
    if configured:
        return configured
    if local_file.exists():
        return local_file.read_text(encoding='utf-8').removesuffix('\n').removesuffix('\r')
    return None

def describe(path):
    if not path.exists():
        return 'assente'
    return f'presente ({path.stat().st_size / 1_000_000:.2f} MB)'

print(f'Repository: {ROOT}')
print(f'Python del kernel: {PYTHON}')
print(f'MC_API_KEY nel kernel: {"configurata" if os.environ.get("MC_API_KEY") else "non rilevata (lo script legge anche .env)"}')
print(f'Deposito Drive: {"configurato" if local_drive_root() else "non configurato"}')
for label, path in [('URL discovery', URLS_FILE), ('Full-text', FULLTEXT_FILE), ('Copertura', COVERAGE_FILE), ('Revisione topic', REVIEW_FILE)]:
    print(f'{label}: {describe(path)}')


In [ ]:
def run_step(label, command):
    print(f'\n=== {label} ===')
    print('Comando:', ' '.join(map(str, command)))
    process = subprocess.Popen(
        command, cwd=ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1
    )
    for line in process.stdout:
        print(line, end='')
    exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f'{label} terminato con codice {exit_code}.')
    print(f'=== {label}: completato ===')


## 1. Discovery Media Cloud

Output atteso: `data/raw/mediacloud_urls.jsonl` e relativa copia su Drive. Il file contiene URL e metadati, non il testo degli articoli. Per procedere, impostare `RUN_DISCOVERY = True` nella configurazione e rieseguire la cella seguente.

In [ ]:
if RUN_DISCOVERY:
    command = [PYTHON, 'src/mediacloud_spike.py']
    if DISCOVERY_MAX_STORIES is not None:
        command.extend(['--max-stories', str(DISCOVERY_MAX_STORIES)])
    run_step('Discovery Media Cloud', command)
    print('Output URL:', describe(URLS_FILE))
else:
    print('Discovery non avviata. Impostare RUN_DISCOVERY = True per eseguirla.')


## 2. Download full-text e pulizia

Richiede l'output della discovery. Scarica il testo dalle fonti originali, conserva lingua e provenienza, elimina gli articoli sportivi entrati dalla query `Lega` e genera la copertura per partito. E' la fase piu' lenta.

In [ ]:
if RUN_FULLTEXT:
    if not URLS_FILE.exists():
        raise FileNotFoundError('Manca mediacloud_urls.jsonl: eseguire prima la discovery.')
    run_step('Download full-text e pulizia', [PYTHON, 'src/mediacloud_fulltext.py'])
    print('Output full-text:', describe(FULLTEXT_FILE))
    print('Output copertura:', describe(COVERAGE_FILE))
else:
    print('Full-text non avviato. Impostare RUN_FULLTEXT = True dopo la discovery.')


## 3. Classificatore dei topic

Richiede il full-text. Stima topic esplorativi con TF-IDF + NMF e crea: (a) un file articolo-per-articolo con pesi e estratti; (b) un file topic con termini caratteristici, da compilare manualmente nei campi `macrotema_validato` e `note_revisione`. Il modello non assegna automaticamente macrotemi finali.

In [ ]:
if RUN_CLASSIFIER:
    if not FULLTEXT_FILE.exists():
        raise FileNotFoundError('Manca mediacloud_fulltext.jsonl: eseguire prima il full-text.')
    run_step('Classificatore topic', [PYTHON, 'src/news_topic_model.py', '--n-topics', str(N_TOPICS)])
    print('Output revisione:', describe(REVIEW_FILE))
    print('Output termini:', describe(TERMS_FILE))
else:
    print('Classificatore non avviato. Impostare RUN_CLASSIFIER = True dopo il full-text.')


## Evoluzione consigliata

Per il progetto attuale, script CLI + notebook di orchestrazione e' un compromesso robusto e trasparente. Per esecuzioni periodiche e osservabilita' di produzione, lo stato dell'arte e' spostare le stesse funzioni in asset Dagster, aggiungere test e metadati di run, e registrare configurazione/modello con MLflow o DVC. La validazione umana del mapping topic → macrotema resta intenzionalmente fuori dall'automazione.